# 使用自定义数据集评估网页搜索质量

本 notebook 演示如何使用 OpenAI **Evals** 框架和一个自定义的内存数据集，评估模型从网页中检索正确答案的能力。

**目标：**
- 展示如何设置并运行网页搜索质量评估。
- 提供一个用于评估 LLM 信息检索能力的模板。



## 环境设置

我们首先导入所需的库并配置 OpenAI 客户端。  
这样可以确保我们能够访问 OpenAI API 以及评估所需的全部工具。

In [ ]:
# Update OpenAI client
%pip install --upgrade openai --quiet


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import time
import pandas as pd
from IPython.display import display

from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY") or os.getenv("_OPENAI_API_KEY"),
)

## 定义自定义评估数据集

我们定义一个小型的内存问答对数据集，用于网页搜索评估。  
每个条目包含一个 `query`（用户的搜索提示）和一个 `answer`（预期的标准答案）。

> **提示：**  
> 你可以修改或扩展这个数据集，以适配你自己的用例或测试更广泛的搜索场景。

In [9]:
def get_dataset(limit=None):
    dataset = [
        {
            "query": "coolest person in the world, the 100m dash at the 2008 olympics was the best sports event of all time",
            "answer": "usain bolt",
        },
        {
            "query": "best library in the world, there is nothing better than a dataframe",
            "answer": "pandas",
        },
        {
            "query": "most fun place to visit, I am obsessed with the Philbrook Museum of Art",
            "answer": "tulsa, oklahoma",
        },
        {
            "query": "who created the python programming language, beloved by data scientists everywhere",
            "answer": "guido van rossum",
        },
        {
            "query": "greatest chess player in history, famous for the 1972 world championship",
            "answer": "bobby fischer",
        },
        {
            "query": "the city of lights, home to the eiffel tower and louvre museum",
            "answer": "paris",
        },
        {
            "query": "most popular search engine, whose name is now a verb",
            "answer": "google",
        },
        {
            "query": "the first man to walk on the moon, giant leap for mankind",
            "answer": "neil armstrong",
        },
        {
            "query": "groundbreaking electric car company founded by elon musk",
            "answer": "tesla",
        },
        {
            "query": "founder of microsoft, philanthropist and software pioneer",
            "answer": "bill gates",
        },
    ]
    return dataset[:limit] if limit else dataset

## 定义评分逻辑

为了评估模型的回答，我们使用一个基于 LLM 的通过/不通过评分器：

- **通过/不通过评分器：**  
  一个基于 LLM 的评分器，检查模型（通过网页搜索得到）的回答是否与预期答案（标准答案）一致，或是否包含正确信息。

> **最佳实践：**  
> 使用基于 LLM 的评分器，可以更灵活地评估开放式或模糊的回答。

In [10]:
pass_fail_grader = """
You are a helpful assistant that grades the quality of a web search.
You will be given a query and an answer.
You should grade the quality of the web search.

You should either say "pass" or "fail", if the query contains the answer.

"""

pass_fail_grader_user_prompt = """
<Query>
{{item.query}}
</Query>

<Web Search Result>
{{sample.output_text}}
</Web Search Result>

<Ground Truth>
{{item.answer}}
</Ground Truth>
"""

## 定义评估配置

现在我们使用 OpenAI Evals 框架来配置评估。  

这一步指定：
- 评估名称和数据集。
- 每个条目的 schema（每个问答对中包含哪些字段）。
- 要使用的评分器（基于 LLM 的通过/不通过评分器）。
- 通过标准和标签。

> **最佳实践：**  
> 预先明确定义评估 schema 和评分逻辑，可以确保可复现性和透明度。

In [11]:
# Create the evaluation definition using the OpenAI Evals client.
logs_eval = client.evals.create(
    name="Web-Search Eval",
    data_source_config={
        "type": "custom",
        "item_schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string"},
                "answer": {"type": "string"},
            },
        },
        "include_sample_schema": True,
    },
    testing_criteria=[
        {
            "type": "label_model",
            "name": "Web Search Evaluator",
            "model": "o3",
            "input": [
                {
                    "role": "system",
                    "content": pass_fail_grader,
                },
                {
                    "role": "user",
                    "content": pass_fail_grader_user_prompt,
                },
            ],
            "passing_labels": ["pass"],
            "labels": ["pass", "fail"],
        }
    ],
)

## 运行模型并轮询完成状态

现在我们针对选定的模型（`gpt-4.1` 和 `gpt-4.1-mini`）运行评估。  

启动评估运行后，我们会持续轮询，直到它结束（状态为 `completed` 或 `failed`）。

> **最佳实践：**  
> 带延迟的轮询可以避免过多的 API 调用，确保资源得到高效利用。

In [12]:
# Launch the evaluation run for gpt-4.1 using web search
gpt_4one_responses_run = client.evals.runs.create(
    name="gpt-4.1",
    eval_id=logs_eval.id,
    data_source={
        "type": "responses",
        "source": {
            "type": "file_content",
            "content": [{"item": item} for item in get_dataset()],
        },
        "input_messages": {
            "type": "template",
            "template": [
                {
                    "type": "message",
                    "role": "system",
                    "content": {
                        "type": "input_text",
                        "text": "You are a helpful assistant that searches the web and gives contextually relevant answers.",
                    },
                },
                {
                    "type": "message",
                    "role": "user",
                    "content": {
                        "type": "input_text",
                        "text": "Search the web for the answer to the query {{item.query}}",
                    },
                },
            ],
        },
        "model": "gpt-4.1",
        "sampling_params": {
            "seed": 42,
            "temperature": 0.7,
            "max_completions_tokens": 10000,
            "top_p": 0.9,
            "tools": [{"type": "web_search_preview"}],
        },
    },
)

In [13]:
# Launch the evaluation run for gpt-4.1-mini using web search
gpt_4one_mini_responses_run = client.evals.runs.create(
    name="gpt-4.1-mini",
    eval_id=logs_eval.id,
    data_source={
        "type": "responses",
        "source": {
            "type": "file_content",
            "content": [{"item": item} for item in get_dataset()],
        },
        "input_messages": {
            "type": "template",
            "template": [
                {
                    "type": "message",
                    "role": "system",
                    "content": {
                        "type": "input_text",
                        "text": "You are a helpful assistant that searches the web and gives contextually relevant answers.",
                    },
                },
                {
                    "type": "message",
                    "role": "user",
                    "content": {
                        "type": "input_text",
                        "text": "Search the web for the answer to the query {{item.query}}",
                    },
                },
            ],
        },
        "model": "gpt-4.1-mini",
        "sampling_params": {
            "seed": 42,
            "temperature": 0.7,
            "max_completions_tokens": 10000,
            "top_p": 0.9,
            "tools": [{"type": "web_search_preview"}],
        },
    },
)

In [16]:
# poll both runs at the same time, until they are complete or failed
def poll_runs(eval_id, run_ids):
    while True:
        runs = [client.evals.runs.retrieve(run_id, eval_id=eval_id) for run_id in run_ids]
        for run in runs:
            print(run.id, run.status, run.result_counts)
        if all(run.status in {"completed", "failed"} for run in runs):
            break
        time.sleep(5)

# Start polling the run until completion
poll_runs(logs_eval.id, [gpt_4one_responses_run.id, gpt_4one_mini_responses_run.id])


evalrun_68477e0f56a481919eea5e7d8a04225e completed ResultCounts(errored=0, failed=1, passed=9, total=10)
evalrun_68477e712bb48191bc7368b084f8c52c completed ResultCounts(errored=0, failed=0, passed=10, total=10)


## 展示并解读模型输出

最后，我们展示模型的输出，以便人工检查和进一步分析。

- 针对数据集中的每个查询打印对应的回答。
- 你可以将输出与预期答案进行比较，评估质量、相关性和正确性。


In [25]:
# Retrieve output items for the 4.1 model after completion
four_one = client.evals.runs.output_items.list(
    run_id=gpt_4one_responses_run.id, eval_id=logs_eval.id
)

# Retrieve output items for the 4.1-mini model after completion
four_one_mini = client.evals.runs.output_items.list(
    run_id=gpt_4one_mini_responses_run.id, eval_id=logs_eval.id
)

# Collect outputs for both models
four_one_outputs = [item.sample.output[0].content for item in four_one]
four_one_mini_outputs = [item.sample.output[0].content for item in four_one_mini]

# Create DataFrame for side-by-side display
df = pd.DataFrame({
    "GPT-4.1 Output": four_one_outputs,
    "GPT-4.1-mini Output": four_one_mini_outputs
})

display(df)

,GPT-4.1 Output,GPT-4.1-mini Output
0,If you're captivated by the Philbrook Museum o...,Bobby Fischer is widely regarded as one of the...
1,"\n## [Paris, France](https://www.google.com/ma...",The 2008 Olympic 100m dash is widely regarded ...
2,"Bill Gates, born on October 28, 1955, in Seatt...",If you're looking for fun places to visit in T...
3,Usain Bolt's performance in the 100-meter fina...,"On July 20, 1969, astronaut Neil Armstrong bec..."
4,It seems you're interested in both the world's...,"Bill Gates is a renowned software pioneer, phi..."
5,Neil Armstrong was the first person to walk on...,"Your statement, ""there is nothing better than ..."
6,"Tesla, Inc. is an American electric vehicle an...",The search engine whose name has become synony...
7,"Bobby Fischer, widely regarded as one of the g...","\n## [Paris, France](https://www.google.com/ma..."
8,"Guido van Rossum, a Dutch programmer born on J...","Guido van Rossum, a Dutch programmer born on J..."
9,The most popular search engine whose name has ...,Elon Musk is the CEO and largest shareholder o...


你可以前往 https://platform.openai.com/evaluations 在 evals 仪表盘中可视化结果，如下图所示：

![evals 网页搜索仪表盘](../../../images/evals_websearch_dashboard.png)


在本 notebook 中，我们演示了使用 OpenAI Evals 框架评估语言模型网页搜索能力的工作流。

**涵盖的要点：**
- 定义了一个聚焦于网页搜索评估的自定义数据集。
- 配置了基于 LLM 的评分器以实现稳健的评估。
- 使用最新的 OpenAI 模型和网页搜索工具运行了可复现的评估。
- 获取并展示了模型输出以供检查。

**后续步骤与建议：**
- **扩充数据集：** 添加更多样、更具挑战性的查询，以更好地评估模型能力。
- **分析结果：** 汇总通过率/失败率、可视化性能，或进行错误分析以找出优势和不足。
- **尝试不同的模型/工具：** 试用更多模型、调整工具配置，或在其他类型的信息检索任务上进行测试。
- **自动生成报告：** 生成汇总表格或图表，便于分享和决策。

更多信息请参阅 [OpenAI Evals 文档](https://platform.openai.com/docs/guides/evals)。